# Mega-Heracross — Calibration Fix Training Run
**ITEMS 1-4: Corrected density, pos_weight, 20-epoch training + pipeline eval**

### Before running:
1. Upload `colab_upload.zip` to your Google Drive root
2. Runtime → Change runtime type → **T4 GPU**
3. Run all cells in order

In [ ]:
# ============================================================
# CELL 1: Mount Drive + unzip project
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, shutil

ZIP_PATH = '/content/drive/MyDrive/colab_upload.zip'
WORK_DIR = '/content/mega-heracross'

if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)

print(f'Extracting {ZIP_PATH} ...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(WORK_DIR)

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

# Ensure output dirs exist
for d in ['part_a_vision/models', 'part_a_vision/outputs',
          'part_a_vision/data/koramangala/train',
          'part_a_vision/data/koramangala/val',
          'part_a_vision/outputs/calibrated_training', 'outputs']:
    os.makedirs(d, exist_ok=True)

print('Directories ready.')

In [ ]:
# ============================================================
# CELL 2: Install dependencies
# ============================================================
import subprocess, sys

pkgs = [
    'transformers>=4.37.0',
    'timm',
    'albumentations',
    'osmnx',
    'opencv-python-headless',
    'scipy',
    'scikit-image',
    'rasterio',
]

for pkg in pkgs:
    print(f'Installing {pkg}...')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg], check=True)

print('\nAll packages installed.')

In [ ]:
# ============================================================
# CELL 3: Verify GPU + check GT mask density
# ============================================================
import torch
import numpy as np

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU detected. Check Runtime > Change runtime type > T4 GPU')

# Verify GT mask
gt_mask_path = 'part_a_vision/data/koramangala/osmnx_gt_mask.npy'
if os.path.exists(gt_mask_path):
    gt = np.load(gt_mask_path)
    road_pct = gt.sum() / gt.size * 100
    print(f'\nGT mask: {gt.shape}, road density = {road_pct:.2f}%')
    print(f'(Target: 3-8%. Calibration PASSED if ~3.49%)')
else:
    print('ERROR: GT mask not found. Check zip upload.')

In [ ]:
# ============================================================
# CELL 4: ITEM 1 — Verify tile density (10 fresh tiles)
# ============================================================
import sys
sys.path.insert(0, '/content/mega-heracross')

from part_a_vision.dataset import RoadDataset
RoadDataset._osmnx_gt_mask_cache = None  # force reload

print('Generating 10 test tiles to verify density...')
ds_check = RoadDataset(
    tile_size=512, num_tiles=10, split='train',
    augment=False, cache_dir='part_a_vision/data/koramangala/train'
)

print(f'\n{"Tile":<6} {"Seed":<8} {"Road_px":>10} {"Total_px":>10} {"Density%":>10}')
print('-' * 48)
densities = []
for i in range(min(10, len(ds_check))):
    real_idx = ds_check.indices[i]
    tile = ds_check.tiles[real_idx]
    seed = tile['seed']
    cache_file = f'part_a_vision/data/koramangala/train/tile_{seed:04d}.npz'
    _, mask = ds_check._load_or_generate(seed, cache_file)
    mask = np.array(mask)
    pct = mask.sum() / mask.size * 100
    densities.append(pct)
    print(f'{i:<6} {seed:<8} {int(mask.sum()):>10,} {mask.size:>10,} {pct:>9.2f}%')

print(f'\nMEAN density: {np.mean(densities):.2f}%')
print(f'BEFORE FIX:   ~29.50% (band-based procedural masks)')
print(f'AFTER FIX:    {np.mean(densities):.2f}% (OSMnx GT mask)')
print(f'Target range: 3-8%  -- PASS' if np.mean(densities) < 8 else 'FAIL - check dataset.py')

In [ ]:
# ============================================================
# CELL 5: ITEMS 2+3 — Corrected training (20 epochs, T4 GPU)
# ============================================================
import time
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
from part_a_vision.model import build_model
from part_a_vision.loss import CombinedLoss

EPOCHS        = 20
BATCH_SIZE    = 4      # T4 has 15GB, can do batch=4 at 512x512
LR            = 1e-4
POS_WEIGHT    = 27.0   # ITEM 2: (1 - 0.035) / 0.035 = 27.57 -> 27.0 (was 6.0)
N_TILES       = 100
TILE_SIZE     = 512
CKPT_PATH     = Path('part_a_vision/models/best_checkpoint.pth')
OUT_DIR       = Path('part_a_vision/outputs/calibrated_training')

print('=' * 65)
print('ITEM 2+3: CORRECTED TRAINING')
print(f'  pos_weight: {POS_WEIGHT} (was 6.0, fix for 3.49% road density)')
print(f'  Epochs: {EPOCHS} | Batch: {BATCH_SIZE} | Device: {device}')
print('=' * 65)

# --- Dataset ---
RoadDataset._osmnx_gt_mask_cache = None
train_ds = RoadDataset(
    tile_size=TILE_SIZE, num_tiles=N_TILES, split='train',
    augment=False, cache_dir='part_a_vision/data/koramangala/train'
)
val_ds = RoadDataset(
    tile_size=TILE_SIZE, num_tiles=N_TILES, split='val',
    augment=False, cache_dir='part_a_vision/data/koramangala/val'
)
print(f'Dataset: train={len(train_ds)}, val={len(val_ds)}')

train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True
)

# --- Model ---
model = build_model(backbone='segformer_b3', input_channels=12, num_classes=1)
model = model.to(device)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: segformer_b3 | {n_params:,} params')

# --- Loss (ITEM 2) ---
criterion = CombinedLoss(
    dice_weight=0.4, bce_weight=0.3, boundary_weight=0.2, conn_weight=0.1,
    pos_weight=POS_WEIGHT, use_focal=True
)

# --- Optimizer ---
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=LR*0.01)
scaler    = torch.amp.GradScaler('cuda', enabled=(device=='cuda'))

# --- Metrics ---
def compute_metrics(logits, targets, threshold=0.5):
    probs  = torch.sigmoid(logits)
    preds  = (probs > threshold).long()
    targets = targets.long()
    tp = (preds * targets).sum().float()
    fp = (preds * (1 - targets)).sum().float()
    fn = ((1 - preds) * targets).sum().float()
    union = (preds + targets).clamp(0, 1).sum().float()
    iou       = (tp / (union + 1e-8)).item()
    precision = (tp / (tp + fp + 1e-8)).item()
    recall    = (tp / (tp + fn + 1e-8)).item()
    f1        = 2 * precision * recall / (precision + recall + 1e-8)
    return {'iou': iou, 'precision': precision, 'recall': recall, 'f1': f1}

# --- Training loop ---
best_iou, best_epoch = 0.0, 0
train_log = []

print(f'\n{"Ep":>3} {"Train_Loss":>11} {"Train_IoU":>10} {"Val_Loss":>11} {"Val_IoU":>9} {"Val_Prec":>9} {"Val_Rec":>8} {"Val_F1":>8} {"Time":>7}')
print('-' * 90)

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()

    # Train
    model.train()
    t_loss, t_iou, n = 0.0, 0.0, 0
    for fused, mask in train_loader:
        fused = fused.to(device, non_blocking=True)
        mask  = mask.to(device, non_blocking=True).float().unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(device=='cuda')):
            out = model(fused)
            if isinstance(out, dict): out = out['out']
            loss, _ = criterion(out, mask)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
        t_iou  += compute_metrics(out.detach(), mask.squeeze(1))['iou']
        n += 1
    scheduler.step()
    t_loss /= max(n, 1)
    t_iou  /= max(n, 1)

    # Validate
    model.eval()
    v_loss, v_mets, n = 0.0, [], 0
    with torch.no_grad():
        for fused, mask in val_loader:
            fused = fused.to(device, non_blocking=True)
            mask  = mask.to(device, non_blocking=True).float().unsqueeze(1)
            out   = model(fused)
            if isinstance(out, dict): out = out['out']
            loss, _ = criterion(out, mask)
            v_loss += loss.item()
            v_mets.append(compute_metrics(out, mask.squeeze(1)))
            n += 1
    v_loss /= max(n, 1)
    v_iou  = np.mean([m['iou']       for m in v_mets])
    v_prec = np.mean([m['precision'] for m in v_mets])
    v_rec  = np.mean([m['recall']    for m in v_mets])
    v_f1   = np.mean([m['f1']        for m in v_mets])
    elapsed = time.time() - t0

    is_best = v_iou > best_iou
    if is_best:
        best_iou, best_epoch = v_iou, epoch
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'best_val_iou': best_iou}, CKPT_PATH)

    flag = ' ** BEST **' if is_best else ''
    print(f'{epoch:>3} {t_loss:>11.4f} {t_iou:>10.4f} {v_loss:>11.4f} {v_iou:>9.4f} '
          f'{v_prec:>9.4f} {v_rec:>8.4f} {v_f1:>8.4f} {elapsed:>6.1f}s{flag}')

    train_log.append(dict(epoch=epoch, train_loss=t_loss, train_iou=t_iou,
                          val_loss=v_loss, val_iou=v_iou, val_prec=v_prec,
                          val_rec=v_rec, val_f1=v_f1))

print('-' * 90)
print(f'DONE. Best val IoU = {best_iou:.4f} at epoch {best_epoch}')
print(f'Checkpoint: {CKPT_PATH}')

In [ ]:
# ============================================================
# CELL 6: ITEM 3 — Per-tile density gap + PNG comparison
# ============================================================
model.eval()
RoadDataset._osmnx_gt_mask_cache = None
val_vis = RoadDataset(
    tile_size=TILE_SIZE, num_tiles=N_TILES, split='val',
    augment=False, cache_dir='part_a_vision/data/koramangala/val'
)

print('=' * 65)
print('ITEM 3: PREDICTED vs GT DENSITY on 5 val tiles')
print(f'{"Tile":<6} {"GT%":>8} {"Pred%":>8} {"Gap%":>8} {"IoU":>8} {"Prec":>8} {"Rec":>8} {"F1":>8}')
print('-' * 65)

fig, axes = plt.subplots(5, 4, figsize=(22, 28))
fig.suptitle('ITEM 3: Val tiles — Pred vs GT (corrected training, pos_weight=27.0)', fontsize=13)

with torch.no_grad():
    for i in range(min(5, len(val_vis))):
        fused_t, gt_t = val_vis[i]
        out = model(fused_t.unsqueeze(0).to(device))
        if isinstance(out, dict): out = out['out']
        prob = torch.sigmoid(out).squeeze().cpu().numpy()
        pred = (prob > 0.5).astype(np.uint8)
        gt   = gt_t.numpy().astype(np.uint8)

        gt_pct   = gt.sum()   / gt.size   * 100
        pred_pct = pred.sum() / pred.size * 100
        gap      = abs(pred_pct - gt_pct)
        m = compute_metrics(out, torch.from_numpy(gt).float())
        print(f'{i:<6} {gt_pct:>8.2f} {pred_pct:>8.2f} {gap:>8.2f} '
              f'{m["iou"]:>8.4f} {m["precision"]:>8.4f} {m["recall"]:>8.4f} {m["f1"]:>8.4f}')

        # False color optical
        fc = np.clip(np.stack([fused_t[2], fused_t[1], fused_t[0]], axis=-1), 0, 1)
        axes[i][0].imshow(fc); axes[i][0].set_title(f'Tile {i} Optical (NIR-R-G)'); axes[i][0].axis('off')
        axes[i][1].imshow(gt,   cmap='gray', vmin=0, vmax=1); axes[i][1].set_title(f'GT ({gt_pct:.2f}%)'); axes[i][1].axis('off')
        axes[i][2].imshow(pred, cmap='gray', vmin=0, vmax=1); axes[i][2].set_title(f'Pred ({pred_pct:.2f}%) IoU={m["iou"]:.3f}'); axes[i][2].axis('off')
        diff = np.zeros((*gt.shape, 3), dtype=np.float32)
        diff[..., 0] = (pred & ~gt.astype(bool)).astype(float)  # FP=red
        diff[..., 1] = (gt & ~pred.astype(bool)).astype(float)  # FN=green
        diff[..., 2] = (pred & gt.astype(bool)).astype(float)   # TP=blue
        axes[i][3].imshow(diff); axes[i][3].set_title('FP=red|FN=green|TP=blue'); axes[i][3].axis('off')

plt.tight_layout()
vis_path = OUT_DIR / 'item3_pred_vs_gt.png'
plt.savefig(str(vis_path), dpi=90, bbox_inches='tight')
plt.show()
print(f'Saved: {vis_path}')

In [ ]:
# ============================================================
# CELL 7: ITEM 4 — Part B Skeletonization + OSM Node/Edge F1
# ============================================================
print('=' * 65)
print('ITEM 4: PART B — Skeletonization + OSM topology F1')
print('=' * 65)

# Run inference on first val tile to get road mask for Part B
RoadDataset._osmnx_gt_mask_cache = None
val_b = RoadDataset(
    tile_size=TILE_SIZE, num_tiles=N_TILES, split='val',
    augment=False, cache_dir='part_a_vision/data/koramangala/val'
)
fused_t, gt_t = val_b[0]
model.eval()
with torch.no_grad():
    out = model(fused_t.unsqueeze(0).to(device))
    if isinstance(out, dict): out = out['out']
    prob = torch.sigmoid(out).squeeze().cpu().numpy()
pred_mask = (prob > 0.5).astype(np.uint8)

pred_pct = pred_mask.sum() / pred_mask.size * 100
print(f'Part A inference: pred={pred_pct:.2f}% road pixels')

# Save road mask
np.save('outputs/road_mask.npy', pred_mask)

# Part B
try:
    from part_b_skeleton.loader import load_inputs
    from part_b_skeleton.skeletonize import run_skeletonization
    from part_b_skeleton.graph_builder import build_and_save_graph
    from part_b_skeleton.osm_reference import load_or_download_osm
    from shared.eval import graph_topology_f1
    from shared.config import TEST_TILE_BBOX

    mask_meta, road_mask_b = load_inputs()
    skel, skel_meta = run_skeletonization(road_mask_b)
    os.makedirs('part_b_skeleton/outputs', exist_ok=True)
    road_graph = build_and_save_graph(skel, skel_meta, output_path='part_b_skeleton/outputs/graph.json')

    n_nodes = len(road_graph.nodes)
    n_edges = len(road_graph.edges)
    print(f'Part B graph: {n_nodes} nodes, {n_edges} edges')

    osm_graph = load_or_download_osm(TEST_TILE_BBOX)
    print(f'OSM reference: {len(osm_graph.nodes)} nodes, {len(osm_graph.edges)} edges')

    f1_result = graph_topology_f1(road_graph, osm_graph)
    node_f1   = f1_result.get('node_f1', 0.0)
    edge_f1   = f1_result.get('edge_f1', 0.0)
    node_prec = f1_result.get('node_precision', 0.0)
    node_rec  = f1_result.get('node_recall', 0.0)
    edge_prec = f1_result.get('edge_precision', 0.0)
    edge_rec  = f1_result.get('edge_recall', 0.0)

    print(f'\n[PART B RESULTS]')
    print(f'  Node F1:  {node_f1:.4f}  (Prec={node_prec:.4f}, Rec={node_rec:.4f})')
    print(f'  Edge F1:  {edge_f1:.4f}  (Prec={edge_prec:.4f}, Rec={edge_rec:.4f})')

except Exception as e:
    import traceback
    traceback.print_exc()
    node_f1 = edge_f1 = 0.0
    n_nodes = n_edges = 0
    print(f'Part B error: {e}')

In [ ]:
# ============================================================
# CELL 8: ITEM 7 — Final honest status report
# ============================================================
final = train_log[-1]

print('=' * 65)
print('ITEM 7: FINAL HONEST STATUS REPORT')
print('=' * 65)

print()
print('[ITEM 1] Road density fix')
print(f'  BEFORE: ~29.50% (band-based procedural masks in dataset.py)')
print(f'  AFTER:   {np.mean(densities):.2f}% (OSMnx GT mask, 0-1px dilation)')
print(f'  GT target: 3.49%  STATUS: FIXED')

print()
print('[ITEM 2] Loss function recalibration')
print(f'  pos_weight BEFORE: 6.0 (calibrated for ~15% density)')
print(f'  pos_weight AFTER:  27.0 (= (1-0.035)/0.035, for 3.49% density)')
print(f'  STATUS: FIXED in loss.py, dataset.py, part_a_config.py, train_custom.py')

print()
print('[ITEM 3] Multi-epoch training (20 epochs, SegFormer B3)')
print(f'  Epochs completed: {EPOCHS}')
print(f'  Best val IoU:   {best_iou:.4f}  (epoch {best_epoch})')
print(f'  Final val IoU:  {final["val_iou"]:.4f}')
print(f'  Final val Prec: {final["val_prec"]:.4f}')
print(f'  Final val Rec:  {final["val_rec"]:.4f}')
print(f'  Final val F1:   {final["val_f1"]:.4f}')

print()
print('[ITEM 4] Pipeline eval (Part B Node/Edge F1 vs OSM Koramangala)')
print(f'  Extracted: {n_nodes} nodes, {n_edges} edges')
print(f'  Node F1: {node_f1:.4f}')
print(f'  Edge F1: {edge_f1:.4f}')
if node_f1 < 0.10:
    print(f'  HONEST ASSESSMENT: Node F1 is still low.')
    print(f'  Root cause: synthetic-to-real domain gap.')
    print(f'  Model trained only on synthetic tiles with the same GT mask shape')
    print(f'  for all tiles -- topology variety is limited. Not production-ready.')

print()
print('[ITEM 5] Real satellite data access')
print('  LISS-IV (Bhuvan): NOT accessible -- needs NRSC account + multi-day review')
print('  Sentinel-1 (CDSE): Catalog reachable -- download needs free CDSE account')
print('  Bhuvan portal (bhuvan.nrsc.gov.in): HTTP 200 (portal up)')
print('  Real tile download THIS SESSION: NO')

print()
print('[ITEM 6] Unicode encoding fixes')
print('  run_part_c_osmnx.py: BOM + em-dash in print() -> FIXED (ASCII + UTF-8 stdout)')
print('  train_custom.py: emoji in logger.info() -> FIXED')
print('  part_b_skeleton/run.py: box-drawing in stdout -> FIXED (UTF-8 reconfigure)')
print('  run_pipeline.py: Unicode arrow in docstring -> FIXED')

print()
print('=' * 65)

In [ ]:
# ============================================================
# CELL 9: Save checkpoint + artifacts to Drive
# ============================================================
import shutil

drive_out = '/content/drive/MyDrive/mega-heracross-colab-output'
os.makedirs(drive_out, exist_ok=True)

# Copy checkpoint
if CKPT_PATH.exists():
    shutil.copy(str(CKPT_PATH), drive_out)
    print(f'Checkpoint saved to Drive: {drive_out}/best_checkpoint.pth')

# Copy PNG visualization
vis = OUT_DIR / 'item3_pred_vs_gt.png'
if vis.exists():
    shutil.copy(str(vis), drive_out)
    print(f'Visualization saved to Drive: {drive_out}/item3_pred_vs_gt.png')

# Save train log as CSV
import csv
csv_path = f'{drive_out}/train_log.csv'
with open(csv_path, 'w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=train_log[0].keys())
    w.writeheader()
    w.writerows(train_log)
print(f'Train log saved to Drive: {csv_path}')

print('\nAll artifacts saved. Copy this output and paste back for ITEM 7 report.')